In [ ]:
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
import re

# 檔案路徑
pdf_path = r"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_146(頁數).pdf"

def extract_hiwin_combined_logic(pdf_path):
    all_tables = []
    
    # 1. 使用 fitz (PyMuPDF) 開啟檔案進行文字掃描
    doc_fitz = fitz.open(pdf_path)
    
    # 定義你發現的型號規則
    # 前綴：F, R, PF, OF, DF
    # 系列碼：SV, SW, DV, DW, SI, DI, SH, SC, DC
    prefixes = ['F', 'R', 'PF', 'OF', 'DF']
    series_codes = ['SV', 'SW', 'DV', 'DW', 'SI', 'DI', 'SH', 'SC', 'DC']
    # 建立正則表達式，例如: ^(F|R|PF|OF|DF)(SV|SW|...)$
    pattern = f"^({'|'.join(prefixes)})({'|'.join(series_codes)})$"
    
    # 2. 使用 pdfplumber 開啟檔案進行表格擷取
    with pdfplumber.open(pdf_path) as pdf_plumb:
        for i in range(len(pdf_plumb.pages)):
            # --- 步驟 A: 使用 fitz 抓取並拼湊系列名 ---
            page_fitz = doc_fitz[i]
            text_lines = [line.strip() for line in page_fitz.get_text().split('\n') if line.strip()]
            
            series_name = "Unknown"
            
            try:
                # 1. 定位邊界：從 TYPE 到 型號 之間
                type_idx = next(idx for idx, line in enumerate(text_lines) if "TYPE" in line.upper())
                model_idx = next(idx for idx, line in enumerate(text_lines) if "型號" in line)
                
                collected_chars = []
                for j in range(type_idx + 1, model_idx):
                    line_text = text_lines[j]
                    
                    # 關鍵：如果遇到直徑符號 Ø 或 ø，代表進入尺寸標註區，停止收集
                    if 'Ø' in line_text or 'ø' in line_text:
                        break
                    
                    # 移除所有非大寫字母 (處理 0FSV 或文字黏連)
                    clean_char = re.sub(r'[^A-Z]', '', line_text)
                    
                    # 逐一收集字母
                    if clean_char:
                        # 如果該行本身就是一個完整的型號 (符合規則)，優先採用
                        if re.match(pattern, clean_char):
                            series_name = clean_char
                            break
                        collected_chars.append(clean_char)
                
                # 如果還沒找到，嘗試拼湊結果進行匹配
                if series_name == "Unknown" and collected_chars:
                    combined = "".join(collected_chars)
                    # 在拼湊出來的字串中尋找符合規則的子字串
                    # 例如從 "0FSV" 提取出 "FSV"
                    search_match = re.search(f"({'|'.join(prefixes)})({'|'.join(series_codes)})", combined)
                    if search_match:
                        series_name = search_match.group(0)

                if series_name == "Unknown":
                    series_name = last_known_series
                else:
                    # 更新最後已知的系列名，供下一頁參考
                    last_known_series = series_name 
                        
            except (StopIteration, ValueError):
                # Fallback: 全頁搜尋符合規則的單字
                for line in text_lines[:50]: # 只看前 50 行
                    clean_line = re.sub(r'[^A-Z]', '', line)
                    if re.match(pattern, clean_line):
                        series_name = clean_line
                        break

            # --- 步驟 B: 使用 pdfplumber 擷取表格 ---
            page_plumb = pdf_plumb.pages[i]
            table_settings = {
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 3,
                "join_tolerance": 2,
                "intersection_tolerance": 5,
            }
            
            table = page_plumb.extract_table(table_settings)
            
            if table:
                df = pd.DataFrame(table)
                df = df.replace('\n', '', regex=True)
                
                # 插入系列名與頁碼 (不直接匯出，存入 list)
                df.insert(0, "Series_Name", series_name)
                df.insert(1, "Source_Page", i + 45) 
                all_tables.append(df)
                print(f"第 {i+1} 頁處理完成，偵測到系列: {series_name}")

    return all_tables

# 執行擷取
all_tables = extract_hiwin_combined_logic(pdf_path)

# --- 後續你可以直接在下方預覽第 N 個表格的品質 ---
# if all_tables:
#     display(all_tables[0].head(10))

第 1 頁處理完成，偵測到系列: FSV
第 2 頁處理完成，偵測到系列: FSV
第 3 頁處理完成，偵測到系列: FSV
第 4 頁處理完成，偵測到系列: FSW
第 5 頁處理完成，偵測到系列: FSW
第 6 頁處理完成，偵測到系列: FSW
第 7 頁處理完成，偵測到系列: FDV
第 8 頁處理完成，偵測到系列: FDV
第 9 頁處理完成，偵測到系列: FDV
第 10 頁處理完成，偵測到系列: FDW
第 11 頁處理完成，偵測到系列: FDW
第 12 頁處理完成，偵測到系列: FDW
第 13 頁處理完成，偵測到系列: FSI
第 14 頁處理完成，偵測到系列: FSI
第 15 頁處理完成，偵測到系列: FSI
第 16 頁處理完成，偵測到系列: RSI
第 17 頁處理完成，偵測到系列: Unknown
第 18 頁處理完成，偵測到系列: FDI
第 19 頁處理完成，偵測到系列: FDI
第 20 頁處理完成，偵測到系列: RDI
第 21 頁處理完成，偵測到系列: RDI
第 22 頁處理完成，偵測到系列: Unknown
第 23 頁處理完成，偵測到系列: Unknown
第 24 頁處理完成，偵測到系列: PFDW
第 25 頁處理完成，偵測到系列: PFDW
第 26 頁處理完成，偵測到系列: PFDI
第 27 頁處理完成，偵測到系列: PFDI
第 28 頁處理完成，偵測到系列: OFSW
第 29 頁處理完成，偵測到系列: OFSW
第 30 頁處理完成，偵測到系列: Unknown
第 31 頁處理完成，偵測到系列: OFSI
第 32 頁處理完成，偵測到系列: FSH
第 33 頁處理完成，偵測到系列: DFSV
第 34 頁處理完成，偵測到系列: FSI
第 35 頁處理完成，偵測到系列: Unknown
第 36 頁處理完成，偵測到系列: Unknown
第 37 頁處理完成，偵測到系列: FSI
第 38 頁處理完成，偵測到系列: FSI
第 39 頁處理完成，偵測到系列: FSI
第 40 頁處理完成，偵測到系列: Unknown
第 41 頁處理完成，偵測到系列: Unknown
第 42 頁處理完成，偵測到系列: FSI
第 43 頁處理完成，偵測到系列: FSI
第 44 頁處理完成，偵測到系列: Unkno

In [16]:
### 文本擷取

import fitz  # PyMuPDF
import re


def step1_extract_text(pdf_path):
    """
    從 PDF 中提取文字，並進行初步的格式清理。
    """
    try:
        # 開啟 PDF 檔案
        doc = fitz.open(pdf_path)
        print(f"--- 檔案讀取成功：{pdf_path} ---")
        print(f"總頁數: {len(doc)}")
        
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # 提取文字
            raw_text = page.get_text("text")
            
            # 初步清理：移除多餘的連續空白、統一換行符
            clean_text = re.sub(r'\n\s*\n', '\n\n', raw_text) # 保持段落感
            clean_text = clean_text.strip()
            
            # 儲存結果（包含頁碼資訊，這對後續 RAG 引用非常重要）
            extracted_data.append({
                "page": page_num + 1,
                "content": clean_text
            })
            
            # 預覽前兩頁
            if page_num < 2:
                print(f"\n[第 {page_num + 1} 頁預覽]:")
                print(clean_text[:300] + "...") 
                print("-" * 30)

        doc.close()
        return extracted_data

    except Exception as e:
        print(f"讀取失敗：{e}")
        return None

# --- 執行處 ---
# 請將 'HIWIN_Catalog.pdf' 換成你實際的檔案路徑

raw_pages = step1_extract_text(R"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf")

--- 檔案讀取成功：C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf ---
總頁數: 5

[第 1 頁預覽]:
S99TC13-1304 41
Type
規格品
6.2 精密研磨級滾珠螺桿尺寸
F
S
V
型號
規格
珠徑
PCD
根徑
珠卷數
剛性
Kgf /μm
K
動負荷
C ( kgf )
靜負荷
Co ( kgf )
螺帽
法蘭
 迴流管
法蘭孔
接觸
面長
公稱
外徑
導程
D
L
F
T
BCD-E
W
H
X
Y
Z
S
16-4B2
16       
4
2.381
16.25
13.792
2.5x2
26
802
1722
30
48
52
10
40
23
21
5.5
9.5
5.5
12
16-5B1
5
3.175
16.6
13.324
2.5x1
16
763
140...
------------------------------

[第 2 頁預覽]:
S99TC13-1304
42
Type
規格品
F
S
V
ØF 
ØD 
-0.1 
-0.3 
ØDg6 
30° 
30° 
Wmax 
Hmax 
BCD E 
ØX 
ØY 
L 
Z 
S 
T 
油孔
T<12  M6x1P
T≥12  1/8PT
型號
規格
珠徑
PCD
根徑
珠卷數
剛性
Kgf /μm
K
動負荷
C ( kgf )
靜負荷
Co ( kgf )
螺帽
法蘭
 迴流管
法蘭孔
接觸
面長
公稱
外徑導程
D
L
F
T
BCD-E
W
H
X
Y
Z
S
36-10B2
36
10
6.350
37.4
30.91
2.5x2
68
5105
12669...
------------------------------
